# Workshop Notebook #1 - MIRI/LRS Solutions

## Author: Taylor Bell (ESA/AURA for STScI)

Run the Setup.ipynb notebook to download the data. The setup notebook needs to just be run **once**.

# Installing Eureka! on your computer

You should first follow the installation instructions at https://eurekadocs.readthedocs.io/en/latest/installation.html to install Eureka! on your computer.

---
# Setting up the notebook

The first step is to setup the notebook and environment.

We'll first import Eureka! along with some other useful packages.

In [ ]:
import eureka
import os
import numpy as np

Next, we need to choose a short, meaningful label (without spaces) that describes the data we're currently working on. For simplicity, we will just set `eventlabel = 'miri_lrs'`. This same event label should be used throughout all stages.

In [ ]:
eventlabel = 'miri_lrs'

To save time, we've also provided _rateints files produced from Stage 1 which would otherwise take 15+ minutes to produce (the exact runtime depends on your CPU and on whether multiple CPU threads are used). If you want to use those cached _rateints files, set `use_cached_s1` to `True`, otherwise set it to `False`.

In [ ]:
use_cached_s1 = True

---
# Stage 1: jwst Pipeline (Eureka!'s wrapper)

## Setting the Stage 1 "Eureka! Control File"

Some of the parameters that might be worth varying are as follows:

* `maximum_cores`: If want to limit the CPU usage of the stage, for example if you are running your data reduction on a shared sherver, you can set this to an integer number of cores or one of `'none'` (for single-threaded operations), `'quarter'`, `'half'`, or `'all'`.
* `skip_firstframe`: This might be worth changing between `True`/`False`. The MIRI detectors show marked persistence effects, which manifest as deviations from linearity at the start of each ramp. To mitigate the effect of persistence on the measured countrates, this step flags the first group in every integration, instructing the pipeline to ignore the first group when fitting the ramps. For TSOs, this step is skipped by default (`skip_firstframe = True`), which does not remove the first frame. In theory it'd be best to set this to `False` (which removes the first group, which is known to be noisy), but for integrations with small numbers of groups, it might be worth setting `skip_firstframe` to `True`. It is hard to say for sure without experimenting.
* `skip_lastframe`: This might be worth changing between `True`/`False`. In the case of MIRI detectors, the array is reset during the last frame read in each integration, which results in a reduced level of accumulated counts in the last frame. By default, the pipeline applies this step to all MIRI data (`skip_firstframe = False`), flagging the last group in every integration as bad (as long as the number of groups per integration is greater than 2). This step is strongly recommended for TSOs (`skip_firstframe = False`), given that the last frame effect has been shown to vary from integration to integration. However, for integrations with a very small number of groups it might be worth setting to `True`. It is hard to say for sure without experimenting.
* `skip_emicorr`: It could be worth changing this between `True`/`False`, but this likely won't have a very large impact. MIRI subarrays suffer from electromagnetic interference (EMI) noise patterns in the raw data that imprint periodic noise into each frame image, with the effect particularly apparent in the case of short ramps, which is typical of most MIRI TSOs. There are two algorithms that the pipeline can select from when applying this step: 'sequential' (the default) fits the EMI in the residuals from ramp fits, while 'joint' carries out a simultaneous fit to the ramps and EMI noise using a reference waveform for the EMI oscillations. The 'sequential' method typically requires 10 or more groups per integration for a reliable fit, while the 'joint' method has been demonstrated to successfully fit EMI noise for any ramp length. In practice, the background subtraction we'll do in Stage 3 ends up providing most of the benefits of this step, so it may not always be worth running. By default, the Eureka! pipeline skips this step (`skip_emicorr = True`), while the jwst pipeline's default is to run the step (`skip_emicorr = False`).
* `skip_rscd`: It could be worth changing this between `True`/`False`. This step mitigates nonlinear behavior at the beginning of each ramp due to transient effects on the detector after every reset. While the exact causes of these effects are not fully understood, the severity of the nonlinearity scales with increasing signal, suggesting a type of persistence on the detectors. Currently, this step in the pipeline flags the first N groups at the beginning of all 2nd and higher integrations, where N is fixed to different values for different observing modes (e.g., N=4 in the case of MIRI LRS slitless TSOs). Due to the short ramps of most TSOs, this step is skipped by default (`skip_rscd = True`), as the flagging of a significant fraction of each ramp results in a severe reduction in the signal-to-noise ratio. However, if the TSO includes relatively large-amplitude features (e.g., exoplanet transits), and if the ramps are long enough to allow for the `rscd` step to be run, turning on this step can ameliorate biases in the relative flux levels due to the signal-dependent nonlinearity effects across the time series.
* `skip_jump` & `jump_rejection_threshold`: The `jump` step flags outliers in the ramps due to cosmic ray hits. The algorithm uses an iterative process that examines the countrates between sequential pairs of groups and applies a sigma-clipping filter to flag groups that yield anomalously large countrates. The default sigma threshold used by the pipeline is `4.0`, though in practice, this settings results in a large number of false positives. Users are recommended to adjust the threshold to higher values (e.g., `8`-`10`) or skip this step altogether in favor of outlier masking at later stages of the data processing workflow. The threshold within Eureka! can be set with the `jump_rejection_threshold` parameter, and the step can be skipped by setting `skip_jump` to `True`.
* `skip_dark_current`: By default this step is run for MIRI TSO observations and is likely best left to run (set the skip parameter to `False`), but it's possible better results could be obtained by experimenting with setting the skip parameter to `True`.
* `skip_refpix`: By default this steps is run for MIRI TSO observations that are taken with the FULL array. For subarrays, including the LRS Slitless subarray, this step is automatically skpped and will not run even if set to `False`. It is possible better results could be obtained by experimenting with setting the skip parameters to `True` for data read with the FULL array.

For more information on the options available in Stage 1, visit https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-1

To change a parameter in the metadata class after it has been read-in, you can do something like the following:
```
s1_meta.jump_rejection_threshold = 10.0
```

In [ ]:
s1_ecf_contents = f"""
# Eureka! Control File for Stage 1: Detector Processing

# Stage 1 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-1

suffix                    uncal

pmap                      1364         # Setting a fixed pmap value to ensure consistency in reductions

maximum_cores             'all'        # Options are 'none', 'quarter', 'half', 'all', or any integer

# Pipeline stages
skip_emicorr              True         #### Might be worth experimenting with turning on/off
skip_saturation           False
skip_firstframe           False        #### Might be worth experimenting with turning on/off
skip_lastframe            False        #### Might be worth experimenting with turning on/off
skip_reset                False
skip_linearity            False
skip_rscd                 False        #### Might be worth experimenting with turning on/off
skip_dark_current         False        #### Might be worth experimenting with turning on/off
skip_refpix               False        #### Might be worth experimenting with turning on/off
skip_jump                 False
skip_clean_flicker_noise  True         #### Might be worth experimenting with turning on/off

#Pipeline stages parameters
jump_rejection_threshold  8.0          #### Might be worth experimenting with different values

# Diagnostics
isplots_S1                3
nplots                    5
hide_plots                False
verbose                   True

# Project directory
topdir                    ../miri_lrs/

# Directories relative to project dir
inputdir                  Uncalibrated
outputdir                 Stage1
"""

# This will save the ECF as a file that the next cell can read-in
with open(f'./S1_{eventlabel}.ecf', 'w') as f:
    f.write(s1_ecf_contents)

## Running Stage 1

In [ ]:
if use_cached_s1 or not os.path.exists('../miri_lrs/Stage1/S1_2025-04-18_miri_lrs_run1'):
    s1_meta = eureka.S1_detector_processing.s1_process.rampfitJWST(eventlabel, ecf_path='./')

---
# Stage 2: jwst Pipeline (Eureka!'s wrapper)

## Setting up the Stage 2 "Eureka! Control File"

For MIRI TSO data, there is very little of importance in Stage 2. The main steps of note are:
* `assign_wcs`: This non-optional step is what assigns the wavelength solution to your spectra.
* `flat_field`: This step corrects for the non-uniform illumination and response of different pixels across the detector. Given that the location of the target of interest on the detector is typically extremely stable across the time series, this step often isn't very important for TSO data, as long as you are not concerned with producing absolutely calibrated data.
* `photom`: This step converts the units of the images from countrate units (DN/sec) of astrophysical flux units (MJy/sr). Since the analysis of TSO data generally does not require absolutely calibrated fluxes, we normally skip this step (by setting `skip_photom` to `True`) for exoplanet observations. Doing this has the added advantage of more easily being able to estimate the uncertainties in the time series later on. However, if you want to produce an absolutely calibrated spectrum of the host star in order to compare the observed stellar flux with stellar models, then the `photom` step should be turned on.
* `extract_1d`: This step extracts the stellar spectra and converts the 2D detector image to a 1D spectrum for each integration. The extraction methods used in this step are not ideal for TSO data, and the step can be quite slow, so we typically always skip this step (by setting `skip_extract1d` to `True`) when working on exoplanet TSO data.

In [ ]:
s2_ecf_contents = f"""
# Eureka! Control File for Stage 2: Data Reduction

# Stage 2 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-2

suffix          rateints    # Data file suffix

pmap            1364        # Setting a fixed pmap value to ensure consistency in reductions

# Note: different instruments and modes will use different steps by default
skip_flat_field False       #### Might be worth experimenting with turning on/off
skip_photom     True        # Strongly recommended to skip (unless doing flux calibrated photometry) in order to get better uncertainties out of Stage 3.
skip_extract1d  True        # Strongly recommended to skip (unless doing flux calibrated photometry) to save a great deal of time.

# Diagnostics
hide_plots      False       # If True, plots will automatically be closed rather than popping up

# Project directory
topdir          ../miri_lrs/

# Directories relative to project dir
inputdir        Stage1
outputdir       Stage2
"""

# This will save the ECF as a file that the next cell can read-in
with open(f'./S2_{eventlabel}.ecf', 'w') as f:
    f.write(s2_ecf_contents)

## Running Stage 2

In [ ]:
s2_meta = eureka.S2_calibrations.s2_calibrate.calibrateJWST(eventlabel, ecf_path='./')

---
# Stage 3: Eureka!

### Setting the Stage 3 "Eureka! Control File" (ECF)

**This determines what will happen during Stage 3**

The most important parameters and their recommended settings are described below, but more context can be found on the [Eureka! documentation website](https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-3).

1.   Set `ncpu` to the number of CPU threads you want to use. If set to `1` no multiprocessing will be done, and this parameter can be increased to ~2x your CPU core count for faster runs.
2.   Set `ywindow` so that it captures the region containing the portion of the star's spectrum you want to extract. A reasonable setting might be `[10, 390]` which captures the entire nominal MIRI/LRS wavelength region (5–14 microns), but we'll use `[141,390]` to capture 5–12 micron wavelength range.
3.   Set `xwindow` so that it excludes obviously bad columns (e.g., columns 0–10). Because there is a known linear slope across the MIRI/LRS background, it is important that you either (a) ensure that there are an equal number of background pixels to the left and right of the source and set `bg_deg` equal to `0` or (b) use any reasonable `xwindow` values and set `bg_deg=1` (easier to do, but comes at a penalty of higher noise). A reasonable setting would be `[11, 61]` with `bg_deg` set to `0`.
4.   Choose the `gaussian` centroiding method to be able to measure the spatial position and width of the star's point-spread function (PSF).
5.   Set `record_ypos` to `True` to record the position and width of the star's PSF on the detector for all frames. This will give us useful values to decorrelate against during our light curve analysis.
6.   Set `dqmask` to `True` to mask bad pixels (e.g., cosmic ray hits) identified by the `jwst` pipeline and marked in the data quality (DQ) array of the input _calints files.
7.   Set `ff_outlier` to `False`. If set to `True`, this step would perform sigma clipping along the entire time axis for the entire frame, while this is only done to the background pixels if `ff_outlier` is set to `False`. Setting `ff_outlier` to `True` is only really safe for observations of shallow eclipses where the sigma-clipping algorithm is unlikely to confuse an astrophysical signal for an outlier.
8.   Set `bg_thresh` (background area outlier threshold) values to `[5,5]` to do two iterations of 5-sigma clipping along the time axis to remove artifacts like cosmic rays.
9.   Set `bg_hw` (background exclusion area half-width) to something large enough to not include significant starlight (something between `8` and `15` should do).
10.  Set `bg_deg` according to point #3 above.
11.  Set `p3thresh` to `5` to sigma-clip 5-sigma outliers along the spatial direction. This will help to remove any remaining starlight left.
12.  Set `spec_hw` (spectral aperture half-width) to something large enough to capture much of the starlight (something between `4` and `8` should do).
13.  Set `fittype` to `meddata` to use the median frame (median computed along the time axis) as the profile for the optimal extraction method.
14.  Set `windowlen` to `1` so that no spectral smoothing is applied to the optimal extraction profile (at least for now - you can experiment with smoothing later if you want).
15.  Set `median_thresh` to `5` to clip 5-sigma outliers along the spectral-axis when computing the median frame.
16.  You can safely ignore `prof_deg` (only used if fittype=poly) and `p5thresh` (only used when `fittype` is `smooth`, `poly`, or `gauss`)
17.  Set `p7thresh` to `10` to sigma-clip 10-sigma outlier pixels compared to the optimal profile.
18.  Set `isplots_S3` to `4` to get lots of useful diagnostic figures (increase this to 5 if you need more plots to investigate problems).
19.  Set `nplots` to `5` to make repetitive figures only for the first 5 integrations (you can increase this as needed if you want more figures for troubleshooting)
20.  Set `hide_plots` to `False` so that the figures pop up in this notebook as they're made (set to `True` if you are not running the code in a notebook; otherwise you will have a lot of windows popping up).
21.  Set `verbose` to `True` so you get lots of useful information printed out.
22.  Set `topdir` to the same value as you used in Stage 2

In [ ]:
s3_ecf_contents = f"""
# Eureka! Control File for Stage 3: Data Reduction

ncpu            72          # Set this to the number of CPU threads you want to use to process the data
nfiles          100         # Set this to the number of file segments you want to process simultaneously (more files typically means faster results, but can end up using too much RAM)
max_memory      0.5         # The maximum fraction of your computer's memory you want utilized by the read-in frames (shouldn't set this above 0.5)
indep_batches   False       # Independently treat each batch of files? Strongly recommended to leave this as False unless you have a clear reason to set it to True.
suffix          calints     # Data file suffix

pmap            1364        # Setting a fixed pmap value to ensure consistency in reductions

# Subarray region of interest
ywindow         [141, 390]  # Vertical axis as seen in DS9
xwindow         [11, 61]    # Horizontal axis as seen in DS9
src_pos_type    gaussian    # Determine source position when not given in header (Options: gaussian, weighted, or max)
record_ypos     True        # Option to record the y position and width for each integration (only records if src_pos_type is gaussian)
dqmask          True        # Mask pixels with an odd entry in the DQ array

# Outlier rejection along time axis
ff_outlier		False		# Set False to use only background region (recommended for deep transits)
							# Set True to use full frame (works well for shallow transits/eclipses)
bg_thresh		[5,5]		# Double-iteration X-sigma threshold for outlier rejection along time axis

# Background parameters
bg_hw			11			# Half-width of exclusion region for BG subtraction (relative to source position)
bg_deg			0			# Polynomial order for column-by-column background subtraction, -1 for median of entire frame
p3thresh		5			# X-sigma threshold for outlier rejection during background subtraction

# Spectral extraction parameters
spec_hw         5           # Half-width of aperture region for spectral extraction (relative to source position)
fittype         meddata     # Method for constructing spatial profile (Options: smooth, meddata, poly, gauss, wavelet, or wavelet2D)
window_len      7           # Smoothing window length, when fittype = smooth or meddata (when computing median frame)
median_thresh   5           # Sigma threshold when flagging outliers in median frame, when fittype=meddata and window_len > 1
prof_deg        3           # Polynomial degree, when fittype = poly
p5thresh        10          # X-sigma threshold for outlier rejection while constructing spatial profile
p7thresh        10          # X-sigma threshold for outlier rejection during optimal spectral extraction

# Diagnostics
isplots_S3      4           # Generate few (1), some (3), or many (5) figures (Options: 1 - 5)
nplots          1           # How many of each type of figure do you want to make per file?
vmin            0.98        # Sets the vmin of the color bar for Figure 3101.
vmax            1.01        # Sets the vmax of the color bar for Figure 3101.
time_axis       'y'         # Determines whether the time axis in Figure 3101 is along the y-axis ('y') or the x-axis ('x')
testing_S3      False       # Boolean, set True to only use last file and generate select figures
hide_plots      False       # If True, plots will automatically be closed rather than popping up
save_output     True        # Save outputs for use in S4
save_fluxdata   False       # Save FluxData outputs for debugging or use with other tools (can be quite large files)
verbose         True        # If True, more details will be printed about steps

# Project directory
topdir          ../miri_lrs/

# Directories relative to project dir
inputdir        Stage2
outputdir       Stage3
"""

# This will save the ECF as a file that the next cell can read-in
with open(f'./S3_{eventlabel}.ecf', 'w') as f:
    f.write(s3_ecf_contents)

### Running Eureka!'s Stage 3

The following cell will run Eureka!'s Stage 3 using the settings you defined above. Note that your ECF will be copied to your output folder, making it easy to remember how you produced those outputs hours, days, or years after you reduced the data.

This stage of Eureka! will take ~1 minute to complete for these particular data.

In [ ]:
s3_spec, s3_meta = eureka.S3_data_reduction.s3_reduce.reduce(eventlabel, ecf_path='./')

---
# Stage 4: Eureka!

## Broadband lightcurve

### Setting the Stage 4 "Eureka! Control File" (ECF)

This determines what will happen during Stage 4

The most important parameters and their recommended settings are described below, but more context can be found on the [Eureka! documentation website](https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-4).

1.   Let's first produce just a single white lightcurve by setting `nspecchan` to `1`
2.   Since we're only computing the white lightcurve, set `compute_white` to `False` as this feature is meant for cases where `nspecchan` is greater than `1`.
3.   Let's set the `wave_min` and `wave_max` to the approximate minimum and maximum usable wavelengths from MIRI/LRS. Normally we can use the full 5--12 micron wavelength range, but some unusual systematics affected the >10.5 micron region, so set `wave_min` to `5.0` and `wave_max` to `10.5`.




To sigma-clip outliers, we need to have a reference time-series against which we are comparing our observations. If we just took the whole time-series and sigma clipped compared to the median level of the observations, we may well sigma-clip the entire transit signal in cases where there is a strong transit. Instead, we use a [box-car](https://en.wikipedia.org/wiki/Boxcar_function) filter which acts as a [high-pass filter](https://en.wikipedia.org/wiki/High-pass_filter); this removes any low-frequency signals (e.g. transit ingress/egress, phase variations, linear trends in time) and leaves behind high-frequency noise (cosmic rays, HGA moves, etc.). The most important parameters that control this box-car clipping are `sigma` and `box_width`, but `boundary` and `fill_value` are also relevant parameters.

4.   Remove outliers along the time axis (e.g. cosmic rays) by setting `clip_binned` to `True` (basically always helpful). `clip_unbinned` is typically not helpful and should be set to `False`.
5.   Set `sigma` to a value low enough to clip any obviously errant points while ensuring you are not clipping an excessive number of points and not clipping the transit or eclipse's ingress/egress. For these particular data, something around `4` should do, but this is not a strict rule and will change for each different dataset. The focus here is to remove obviously errant points and not to clip a bunch of points.
6.   Set `box_width` to a value small enough to not sigma-clip the transit or eclipse's ingress/egress, while also not setting it so small that the smoothed copy of the signal is excessively noisy. For these particular data, something around `20` should do, but again this is not a strict rule and will change for each different dataset.
7.   Set `boundary` to `fill` as this typically results in reasonable behaviour at the start and end of the observations.
8.   Set `fill_value` to `mask` in order to mask the clipped outliers without replacement. This ensures you remove bad values without requiring you to replace the masked values with a guess as to the value the point should have (a potentially dangerous endeavour).


Finally, there are some plotting/logging controls you should adjust:

9.  Set `compute_ld` to `False`. Since we're working on an eclipse observation, there is no need to compute theoretical limb-darkening coefficients for the star. The other inputs related to computing limb-darkening coefficients can then safely be ignored.
10.  Set `isplots_S4` to `3` to get some useful diagnostic figures (increase this to 5 if you need more plots to investigate problems).
11.  Set `hide_plots` to `False` so that the figures pop up in this notebook as they're made (set to `True` if you're running the code in the terminal instead of a notebook, otherwise you'll have a lot of windows popping up).
12.  Set `verbose` to `True` so you get lots of useful information printed out.
13.  Set `topdir` to the same value as you used in Stage 3.
14.  Set `inputdir` to `Stage3`. If you end up running multiple version of Eureka!'s Stage 3, you can select the exact one you want as an input to Stage 4 by specifying the folder name in more detail (e.g. `Stage3/S3_2023-07-24_miri_run1`).
15.  Set `outputdir` to `Stage4_white` so that we can distinguish between this white lightcurve from the spectroscopic lightcurves we'll compute later

In [ ]:
s4_ecf_contents = f"""
# Eureka! Control File for Stage 4: Generate Lightcurves

# Number of spectroscopic channels spread evenly over given wavelength range
nspecchan       1       # Number of spectroscopic channels
compute_white   False   # Separately compute the white-light lightcurve?
wave_min        5.0     # Minimum wavelength
wave_max        10.5    # Maximum wavelength
allapers        False   # Run S4 on all of the apertures considered in S3? Otherwise will use newest output in the inputdir

# Parameters for sigma clipping
clip_unbinned   False   # Whether or not sigma clipping should be performed on the 1D time series
clip_binned     True    # Whether or not sigma clipping should be performed on the 1D time series
sigma           4       # The number of sigmas a point must be from the rolling median to be considered an outlier
box_width       20      # The width of the box-car filter (used to calculated the rolling median) in units of number of data points
maxiters        20      # The number of iterations of sigma clipping that should be performed.
boundary        fill    # Use 'fill' to extend the boundary values by the median of all data points (recommended), 'wrap' to use a periodic boundary, or 'extend' to use the first/last data points
fill_value      mask    # Either the string 'mask' to mask the outlier values (recommended), 'boxcar' to replace data with the mean from the box-car filter, or a constant float-type fill value.

# Limb-darkening parameters needed to compute exotic-ld
compute_ld      False

# Diagnostics
isplots_S4      3       # Generate few (1), some (3), or many (5) figures (Options: 1 - 5)
vmin            0.98
vmax            1.01
time_axis       'y'
hide_plots      False   # If True, plots will automatically be closed rather than popping up
verbose         True    # If True, more details will be printed about steps

# Project directory
topdir          ../miri_lrs/

# Directories relative to project dir
inputdir        Stage3
outputdir       Stage4_white
"""

# This will save the ECF as a file that the next cell can read-in
with open(f'S4_{eventlabel}.ecf', 'w') as f:
    f.write(s4_ecf_contents)

### Running Eureka!'s Stage 4

The following cell will run Eureka!'s Stage 4 using the settings you defined above. Note that your ECF will be copied to your output folder, making it easy to remember how you produced those outputs hours, days, or years after you reduced the data.

This stage of Eureka! will take &lt;&lt;1 minute to complete for these particular data.

In [ ]:
s4_spec, s4_lc, s4_meta = eureka.S4_generate_lightcurves.s4_genLC.genlc(eventlabel, ecf_path='./')

---
# Bonus: Getting the "best" reduction

Seeking the "best" reduction is where we venture into the art of data reduction, with various parameters sometimes having appreciable impacts on the precision we can get from our observations and (frighteningly) sometimes significantly impacting our final planetary spectra. At all times, think critically about the settings you are using and the potential for unintended consequences.

If you want to try to get even better results, focus on trying different values in Stage 3 for:

**Most impactful:**
*   `spec_hw` (with a larger aperture you capture more of the star's light but also capture more noisy background light - there is a trade-off here which will likely change for every different target you observe)
*   `bg_hw` (with a smaller value you have more background pixels but also more stellar contamination - there is a trade-off here which will likely change for every different target you observe)

**Intermediate impact:**
*   `p3thresh` (the best value will likely change depending on the value you use for `bg_hw`)
*   `p7thresh` (try larger/smaller values)
*   `window_len` (try values roughly in the range of `2`–`11` - larger values may help to smooth over noise in your profile, but if the value is too large it can result in bad extraction profiles and potentially biased results)
*   `median_thresh` (try larger/smaller values - note that this is only used if `fittype` is set to `meddata` and `window_len` is larger than `1`).

**Uncertain impact:**
*   `bg_thresh` (you likely won't see large changes, but might slightly improve the overall scatter)
*   `fittype` (try `smooth`)

<br/>

Typically the "best" value for each parameter is independent of the values of other parameters, but sometimes there are interactions (e.g. a smaller `bg_hw` may require smaller `p3thresh`). The Stage 4 MAD (Median Absolute Deviation) value is an approximate tool you can use to determine how much your reduction has improved (typically the lower the value, the lower the noise in your lightcurve, and the "better" the reduction). A better indication of an improved reduction is the RMS of the residuals from your Stage 5 fit since the Stage 4 MAD doesn't account for noise that can be decorrelated when fitting, but depending on your dataset it can be prohibitively time-intensive to continuously run all reduction options through Stage 5.

Again though, remain critical and vigilant at all times when optimizing your reduction. For example, you'll be able to decrease your MAD value by setting your Stage 4 `sigma` value to `0.01`, but you're going to end up discarding nearly all of your data which you definitely don't want to do. A less obvious danger might be setting Stage 3's `bg_hw` parameter to too small a value, resulting in self-subtraction and potentially biased transit or eclipse depths.